# 🌊 Surf Spot Evaluation from NOAA Spectral Data
This notebook parses and visualizes spectral wave data from NOAA NDBC buoy `51002`, and helps determine conditions for **good surf**.

Uploaded files used:
- `51002.data_spec` – Energy per frequency (E(f))
- `51002.swdir` – Wave direction (a1 moment)
- `51002.swr1` – Wave direction (b1 moment)

---

## 1️⃣ Load and Parse Spectral Data

In [7]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Plotting config
plt.style.use('default')
%matplotlib inline

In [10]:
# Load raw spectral energy data (E(f))
spec_df = pd.read_csv('51002.data_spec', delim_whitespace=True, header=None)
spec_df.columns = ['YY', 'MM', 'DD', 'hh', 'mm'] + [f'f{i}' for i in range(1, spec_df.shape[1]-5+1)]
spec_df['datetime'] = pd.to_datetime(spec_df[['YY', 'MM', 'DD', 'hh', 'mm']])
spec_df.set_index('datetime', inplace=True)
spec_df = spec_df.drop(columns=['YY', 'MM', 'DD', 'hh', 'mm'])
spec_df.head()

/var/folders/xj/qvjb5ty104s3bjl9tynd9f800000gn/T/ipykernel_83855/4028844594.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  spec_df = pd.read_csv('51002.data_spec', delim_whitespace=True, header=None)


ParserError: Error tokenizing data. C error: Expected 15 fields in line 2, saw 98


In [ ]:
# Load wave direction (alpha1)
swdir_df = pd.read_csv('/mnt/data/51002.swdir', delim_whitespace=True, header=None)
swdir_df.columns = ['YY', 'MM', 'DD', 'hh', 'mm'] + [f'f{i}' for i in range(1, swdir_df.shape[1]-5+1)]
swdir_df['datetime'] = pd.to_datetime(swdir_df[['YY', 'MM', 'DD', 'hh', 'mm']])
swdir_df.set_index('datetime', inplace=True)
swdir_df = swdir_df.drop(columns=['YY', 'MM', 'DD', 'hh', 'mm'])
swdir_df.head()

In [ ]:
# Load directionality (r1 / b1)
swr1_df = pd.read_csv('/mnt/data/51002.swr1', delim_whitespace=True, header=None)
swr1_df.columns = ['YY', 'MM', 'DD', 'hh', 'mm'] + [f'f{i}' for i in range(1, swr1_df.shape[1]-5+1)]
swr1_df['datetime'] = pd.to_datetime(swr1_df[['YY', 'MM', 'DD', 'hh', 'mm']])
swr1_df.set_index('datetime', inplace=True)
swr1_df = swr1_df.drop(columns=['YY', 'MM', 'DD', 'hh', 'mm'])
swr1_df.head()

## 2️⃣ Visualize Spectral Energy Distribution

In [ ]:
# Plot energy spectrum for latest observation
latest = spec_df.iloc[-1]
freqs = np.linspace(0.02, 0.5, len(latest))
plt.figure(figsize=(10, 5))
plt.plot(freqs, latest.values)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Energy (m^2/Hz)')
plt.title(f'Wave Energy Spectrum – {latest.name}')
plt.grid(True)
plt.show()

## 3️⃣ Calculate Significant Wave Height and Peak Period

In [ ]:
# Estimate significant wave height (Hs) from spectral moment m0
df = spec_df.copy()
df['m0'] = df.apply(lambda row: np.trapz(row.values, freqs), axis=1)
df['Hs'] = 4 * np.sqrt(df['m0'])

# Estimate peak frequency and period
df['fp'] = spec_df.apply(lambda row: freqs[np.argmax(row.values)], axis=1)
df['Tp'] = 1 / df['fp']
df[['Hs', 'Tp']].tail()

## 4️⃣ Evaluate Surf Quality Logic

In [ ]:
# Simple logic for 'good surf'
def is_good_surf(hs, tp):
    return hs > 1.2 and tp > 10  # meters, seconds

df['good_surf'] = df.apply(lambda row: is_good_surf(row['Hs'], row['Tp']), axis=1)

# Count how many good surf observations
df['good_surf'].value_counts()

## 5️⃣ Plot Time Series of Hs and Tp with Surf Quality

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))
df['Hs'].plot(ax=ax1, color='blue', label='Significant Wave Height (Hs)')
df['Tp'].plot(ax=ax1, color='green', secondary_y=True, label='Peak Period (Tp)')

# Highlight 'good surf' points
good_times = df[df['good_surf']].index
for gt in good_times:
    ax1.axvline(gt, color='red', linestyle='--', alpha=0.3)

ax1.set_title('Wave Height & Period Over Time with Good Surf Flags')
ax1.set_xlabel('Datetime')
ax1.set_ylabel('Hs (m)')
ax1.right_ax.set_ylabel('Tp (s)')
plt.tight_layout()
plt.show()

## ✅ Summary
This notebook demonstrates how to:
- Parse raw NOAA buoy spectral data
- Compute wave energy, Hs, Tp
- Identify favorable surf conditions

Use this as a foundation for building alert systems, surf prediction, or feeding into a surf logging backend.